In [0]:
# Updated Blob 1: deterministic CDF worklist using committed Delta-version checkpoints.

from datetime import datetime, timezone
import re
import uuid

from delta.tables import DeltaTable
from pyspark.sql import Row, functions as F
from pyspark.sql.window import Window


In [0]:
dbutils.widgets.text("RUN_ID", "")
dbutils.widgets.text("PIPELINE_ID", "mill_blob_text_barts_v3")
dbutils.widgets.text("SOURCE_TABLE", "4_prod.raw.mill_ce_blob")
dbutils.widgets.text("TARGET_TABLE", "4_prod.bronze.mill_blob_text")
dbutils.widgets.text("STAGING_DB", "4_prod.tmp")
dbutils.widgets.text("METADATA_TABLE", "4_prod.tmp.blob_pipeline_metadata_v3")
dbutils.widgets.text("CHECKPOINT_TABLE", "4_prod.tmp.blob_pipeline_checkpoint_v3")
dbutils.widgets.text("TRUST_FILTER", "Barts")
dbutils.widgets.text("MAX_EVENTS", "250000")
dbutils.widgets.text("INITIAL_START_VERSION", "")
dbutils.widgets.dropdown("OPTIMIZE_WORKLIST", "false", ["false", "true"])

RUN_ID = dbutils.widgets.get("RUN_ID").strip() or (
    datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_") + uuid.uuid4().hex[:8]
)
PIPELINE_ID = dbutils.widgets.get("PIPELINE_ID").strip()
SOURCE_TABLE = dbutils.widgets.get("SOURCE_TABLE").strip()
TARGET_TABLE = dbutils.widgets.get("TARGET_TABLE").strip()
STAGING_DB = dbutils.widgets.get("STAGING_DB").strip()
METADATA_TABLE = dbutils.widgets.get("METADATA_TABLE").strip()
CHECKPOINT_TABLE = dbutils.widgets.get("CHECKPOINT_TABLE").strip()
TRUST_FILTER = dbutils.widgets.get("TRUST_FILTER").strip()
MAX_EVENTS = int(dbutils.widgets.get("MAX_EVENTS") or "0")
INITIAL_START_VERSION_RAW = dbutils.widgets.get("INITIAL_START_VERSION").strip()
OPTIMIZE_WORKLIST = dbutils.widgets.get("OPTIMIZE_WORKLIST").lower() == "true"


_TABLE_RE = re.compile(r"^[A-Za-z0-9_]+(?:\.[A-Za-z0-9_]+){1,2}$")
_VALUE_RE = re.compile(r"^[A-Za-z0-9_.:-]+$")


def validate_table_name(value: str) -> str:
    if not _TABLE_RE.fullmatch(value):
        raise ValueError(f"Unsafe table identifier: {value!r}")
    return value


def quote_table(value: str) -> str:
    validate_table_name(value)
    return ".".join(f"`{part}`" for part in value.split("."))


for table_name in (SOURCE_TABLE, TARGET_TABLE, STAGING_DB, METADATA_TABLE, CHECKPOINT_TABLE):
    validate_table_name(table_name)
if not _VALUE_RE.fullmatch(RUN_ID) or not _VALUE_RE.fullmatch(PIPELINE_ID):
    raise ValueError("RUN_ID and PIPELINE_ID may contain only letters, numbers, _, ., :, and -")
if not TRUST_FILTER:
    raise ValueError("TRUST_FILTER cannot be empty")
if MAX_EVENTS < 0:
    raise ValueError("MAX_EVENTS must be zero (unlimited) or positive")

safe_run_id = re.sub(r"[^A-Za-z0-9_]", "_", RUN_ID)
WORKLIST_TABLE = f"{STAGING_DB}.blob_worklist_v3_{safe_run_id}"


In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {quote_table(METADATA_TABLE)} (
  run_id STRING NOT NULL,
  pipeline_id STRING NOT NULL,
  worklist_table STRING,
  source_table STRING NOT NULL,
  target_table STRING NOT NULL,
  trust_filter STRING NOT NULL,
  source_start_version BIGINT,
  source_end_version BIGINT,
  latest_source_version BIGINT,
  detected_event_versions BIGINT,
  total_events BIGINT,
  processed_events BIGINT,
  merged_events BIGINT,
  has_more BOOLEAN,
  status STRING NOT NULL,
  batch_tables STRING,
  history_tables STRING,
  error_message STRING,
  created_ts TIMESTAMP,
  completed_ts TIMESTAMP,
  merged_ts TIMESTAMP
) USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {quote_table(CHECKPOINT_TABLE)} (
  pipeline_id STRING NOT NULL,
  source_table STRING NOT NULL,
  trust_filter STRING NOT NULL,
  last_source_version BIGINT NOT NULL,
  last_run_id STRING,
  updated_ts TIMESTAMP
) USING DELTA
""")

if spark.table(METADATA_TABLE).filter(F.col("run_id") == RUN_ID).limit(1).count():
    raise RuntimeError(f"RUN_ID already exists in metadata: {RUN_ID}")


def append_metadata(**values):
    schema = spark.table(METADATA_TABLE).schema
    payload = {field.name: None for field in schema.fields}
    payload.update(values)
    spark.createDataFrame([Row(**payload)], schema=schema).write.mode("append").insertInto(METADATA_TABLE)


def advance_checkpoint(version: int, run_id: str):
    schema = spark.table(CHECKPOINT_TABLE).schema
    row = Row(
        pipeline_id=PIPELINE_ID,
        source_table=SOURCE_TABLE,
        trust_filter=TRUST_FILTER,
        last_source_version=int(version),
        last_run_id=run_id,
        updated_ts=datetime.now(timezone.utc).replace(tzinfo=None),
    )
    source = spark.createDataFrame([row], schema=schema)
    condition = (
        "t.pipeline_id = s.pipeline_id AND t.source_table = s.source_table "
        "AND t.trust_filter = s.trust_filter"
    )
    (DeltaTable.forName(spark, CHECKPOINT_TABLE).alias("t")
        .merge(source.alias("s"), condition)
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())


In [0]:
history = spark.sql(f"DESCRIBE HISTORY {quote_table(SOURCE_TABLE)}").select(
    F.col("version").cast("long").alias("version"), "timestamp"
)
version_bounds = history.agg(
    F.min("version").alias("earliest"), F.max("version").alias("latest")
).collect()[0]
earliest_version = int(version_bounds["earliest"])
latest_version = int(version_bounds["latest"])

checkpoint_rows = (
    spark.table(CHECKPOINT_TABLE)
    .filter(
        (F.col("pipeline_id") == PIPELINE_ID)
        & (F.col("source_table") == SOURCE_TABLE)
        & (F.col("trust_filter") == TRUST_FILTER)
    )
    .orderBy(F.col("updated_ts").desc_nulls_last())
    .limit(1)
    .collect()
)

if checkpoint_rows:
    start_version = int(checkpoint_rows[0]["last_source_version"]) + 1
elif INITIAL_START_VERSION_RAW:
    start_version = int(INITIAL_START_VERSION_RAW)
else:
    # Safe first-run behavior: replay all CDF still retained, with the target anti-join
    # preventing duplicate event versions.
    start_version = earliest_version

if start_version < earliest_version:
    raise RuntimeError(
        f"Requested CDF start version {start_version} predates retained history "
        f"({earliest_version}). Restore CDF history or explicitly choose a safe bootstrap."
    )

print(
    f"RUN_ID={RUN_ID} source versions={start_version}..{latest_version} "
    f"checkpoint={'present' if checkpoint_rows else 'bootstrap'}"
)

now = datetime.now(timezone.utc).replace(tzinfo=None)
if start_version > latest_version:
    append_metadata(
        run_id=RUN_ID, pipeline_id=PIPELINE_ID, worklist_table=None,
        source_table=SOURCE_TABLE, target_table=TARGET_TABLE,
        trust_filter=TRUST_FILTER, source_start_version=start_version,
        source_end_version=latest_version, latest_source_version=latest_version,
        detected_event_versions=0, total_events=0, processed_events=0,
        merged_events=0, has_more=False, status="no_work", created_ts=now,
        completed_ts=now, merged_ts=now,
    )
    dbutils.jobs.taskValues.set(key="RUN_ID", value=RUN_ID)
    dbutils.jobs.taskValues.set(key="HAS_WORK", value="false")
    dbutils.notebook.exit("NO_WORK")


In [0]:
def read_cdf_range(start: int, end: int):
    frame = (
        spark.read.format("delta")
        .option("readChangeFeed", "true")
        .option("startingVersion", int(start))
        .option("endingVersion", int(end))
        .table(SOURCE_TABLE)
    )
    # Force analysis here so schema-boundary failures are caught by the bisection
    # wrapper instead of surfacing much later during worklist construction.
    _ = frame.schema
    return frame


def read_cdf_resilient(start: int, end: int):
    """Bisect only when a range crosses an incompatible schema boundary."""
    try:
        return read_cdf_range(start, end)
    except Exception as exc:
        if start >= end:
            raise RuntimeError(
                f"CDF version {start} is unreadable; refusing timestamp fallback: {exc}"
            ) from exc
        midpoint = (start + end) // 2
        left = read_cdf_resilient(start, midpoint)
        right = read_cdf_resilient(midpoint + 1, end)
        return left.unionByName(right, allowMissingColumns=True)


cdf = read_cdf_resilient(start_version, latest_version)
required_columns = {
    "EVENT_ID", "BLOB_SEQ_NUM", "VALID_UNTIL_DT_TM", "VALID_FROM_DT_TM",
    "UPDT_DT_TM", "UPDT_ID", "UPDT_TASK", "UPDT_CNT", "UPDT_APPLCTX",
    "LAST_UTC_TS", "ADC_UPDT", "COMPRESSION_CD", "BLOB_CONTENTS",
    "BLOB_LENGTH", "ENCNTR_ID", "Trust", "_change_type", "_commit_version",
    "_commit_timestamp",
}
missing_columns = sorted(required_columns - set(cdf.columns))
if missing_columns:
    raise RuntimeError(f"CDF is missing required columns: {missing_columns}")

changes = (
    cdf.filter(F.col("_change_type").isin("insert", "update_postimage"))
    .filter(F.col("Trust") == TRUST_FILTER)
    .select(
        F.col("EVENT_ID").cast("long").alias("EVENT_ID"),
        F.col("BLOB_SEQ_NUM").cast("long").alias("BLOB_SEQ_NUM"),
        "VALID_UNTIL_DT_TM", "VALID_FROM_DT_TM", "UPDT_DT_TM",
        F.col("UPDT_ID").cast("long").alias("UPDT_ID"),
        F.col("UPDT_TASK").cast("long").alias("UPDT_TASK"),
        F.col("UPDT_CNT").cast("long").alias("UPDT_CNT"),
        F.col("UPDT_APPLCTX").cast("long").alias("UPDT_APPLCTX"),
        "LAST_UTC_TS", "ADC_UPDT",
        F.col("COMPRESSION_CD").cast("long").alias("COMPRESSION_CD"),
        "BLOB_CONTENTS",
        F.col("BLOB_LENGTH").cast("long").alias("BLOB_LENGTH"),
        F.col("ENCNTR_ID").cast("long").alias("ENCNTR_ID"),
        "Trust",
        F.col("_commit_version").cast("long").alias("_commit_version"),
        "_commit_timestamp",
    )
)

# Keep one postimage per exact event-version chunk. ADC_UPDT is deliberately part
# of the key so intermediate source versions are not collapsed.
dedupe_window = Window.partitionBy("EVENT_ID", "ADC_UPDT", "BLOB_SEQ_NUM").orderBy(
    F.col("_commit_version").desc(),
    F.col("UPDT_DT_TM").desc_nulls_last(),
    F.col("LAST_UTC_TS").desc_nulls_last(),
)
deduped = (
    changes.withColumn("_rank", F.row_number().over(dedupe_window))
    .filter(F.col("_rank") == 1)
    .drop("_rank")
)

event_versions = deduped.groupBy("EVENT_ID", "ADC_UPDT").agg(
    F.max("_commit_version").alias("source_commit_version")
)
target_keys = spark.table(TARGET_TABLE).select("EVENT_ID", "ADC_UPDT").distinct()
new_event_versions = event_versions.alias("s").join(
    target_keys.alias("t"),
    (F.col("s.EVENT_ID") == F.col("t.EVENT_ID"))
    & F.col("s.ADC_UPDT").eqNullSafe(F.col("t.ADC_UPDT")),
    "left_anti",
)

detected_count = new_event_versions.count()
if detected_count == 0:
    advance_checkpoint(latest_version, RUN_ID)
    append_metadata(
        run_id=RUN_ID, pipeline_id=PIPELINE_ID, worklist_table=None,
        source_table=SOURCE_TABLE, target_table=TARGET_TABLE,
        trust_filter=TRUST_FILTER, source_start_version=start_version,
        source_end_version=latest_version, latest_source_version=latest_version,
        detected_event_versions=0, total_events=0, processed_events=0,
        merged_events=0, has_more=False, status="no_work", created_ts=now,
        completed_ts=now, merged_ts=now,
    )
    dbutils.jobs.taskValues.set(key="RUN_ID", value=RUN_ID)
    dbutils.jobs.taskValues.set(key="HAS_WORK", value="false")
    dbutils.notebook.exit("NO_WORK")


In [0]:
# Apply the cap only at complete Delta commit boundaries. If a single commit is
# larger than MAX_EVENTS it is processed whole, avoiding a permanently split cursor.
processed_end_version = latest_version
if MAX_EVENTS > 0 and detected_count > MAX_EVENTS:
    version_counts = (
        new_event_versions.groupBy("source_commit_version").count()
        .orderBy("source_commit_version")
        .collect()
    )
    cumulative = 0
    included_any = False
    for row in version_counts:
        version = int(row["source_commit_version"])
        count = int(row["count"])
        if included_any and cumulative + count > MAX_EVENTS:
            processed_end_version = version - 1
            break
        cumulative += count
        included_any = True
        processed_end_version = version
        if cumulative >= MAX_EVENTS:
            break

selected_keys = new_event_versions.filter(
    F.col("source_commit_version") <= F.lit(processed_end_version)
)
has_more = processed_end_version < latest_version

selected_chunks = deduped.alias("d").join(
    selected_keys.alias("k"),
    (F.col("d.EVENT_ID") == F.col("k.EVENT_ID"))
    & F.col("d.ADC_UPDT").eqNullSafe(F.col("k.ADC_UPDT")),
    "inner",
).select("d.*", "k.source_commit_version")

chunk_struct = F.struct(
    "BLOB_SEQ_NUM", "VALID_UNTIL_DT_TM", "VALID_FROM_DT_TM", "UPDT_DT_TM",
    "UPDT_ID", "UPDT_TASK", "UPDT_CNT", "UPDT_APPLCTX", "LAST_UTC_TS",
    "ADC_UPDT", "COMPRESSION_CD", "BLOB_CONTENTS", "BLOB_LENGTH",
    "ENCNTR_ID", "Trust",
)

worklist = (
    selected_chunks.groupBy("EVENT_ID", "ADC_UPDT", "source_commit_version")
    .agg(
        F.sum(F.length("BLOB_CONTENTS")).cast("long").alias("compressed_size"),
        F.count("*").cast("long").alias("chunk_count"),
        F.collect_list(chunk_struct).alias("chunks_data"),
    )
    .withColumn(
        "status",
        F.when(F.col("compressed_size") > 16 * 1024 * 1024, "compressed_oversized")
        .otherwise("pending"),
    )
    .withColumn("process_ts", F.lit(None).cast("timestamp"))
    .withColumn("error_msg", F.lit(None).cast("string"))
)

worklist_count = worklist.count()
if worklist_count == 0:
    raise RuntimeError("Detected event versions but materialized an empty worklist")

temp_view = f"blob_worklist_view_{safe_run_id}"
worklist.createOrReplaceTempView(temp_view)

try:
      spark.sql(f"""
          CREATE TABLE {quote_table(WORKLIST_TABLE)}
          USING DELTA
          AS SELECT * FROM `{temp_view}`
      """)
finally:
      spark.catalog.dropTempView(temp_view)

if OPTIMIZE_WORKLIST and worklist_count >= 100_000:
    spark.sql(
        f"OPTIMIZE {quote_table(WORKLIST_TABLE)} ZORDER BY (source_commit_version, EVENT_ID)"
    )

append_metadata(
    run_id=RUN_ID, pipeline_id=PIPELINE_ID, worklist_table=WORKLIST_TABLE,
    source_table=SOURCE_TABLE, target_table=TARGET_TABLE,
    trust_filter=TRUST_FILTER, source_start_version=start_version,
    source_end_version=processed_end_version, latest_source_version=latest_version,
    detected_event_versions=detected_count, total_events=worklist_count,
    processed_events=0, merged_events=0, has_more=has_more,
    status="worklist_created", created_ts=now,
)

dbutils.jobs.taskValues.set(key="RUN_ID", value=RUN_ID)
dbutils.jobs.taskValues.set(key="HAS_WORK", value="true")
print(
    f"Created {WORKLIST_TABLE}: {worklist_count:,} event versions, "
    f"source versions {start_version}..{processed_end_version}, has_more={has_more}"
)